# Log google/medsiglip-448 to Snowflake Model Registry
Dual-encoder CustomModel for DICOM image + text embeddings, deployed as SPCS inference service on GPU.

*Co-authored with CoCo*

In [ ]:
%%sql
USE SCHEMA SF_CLINICAL_DB.UTILS;

In [ ]:
!pip install transformers torch pillow huggingface_hub pydicom --quiet

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("""
CREATE OR REPLACE FUNCTION SF_CLINICAL_DB.UTILS.GET_HF_TOKEN()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'get_token'
EXTERNAL_ACCESS_INTEGRATIONS = (ALLOW_ALL_EAI)
SECRETS = ('hf_token' = CORTEX_AI_DB.UTILS.SECRET_NAME_HUGGING_FACE_TOKEN)
AS $$
import _snowflake
def get_token():
    return _snowflake.get_generic_secret_string('hf_token')
$$
""").collect()

HF_TOKEN = session.sql("SELECT SF_CLINICAL_DB.UTILS.GET_HF_TOKEN() AS TOKEN").collect()[0]['TOKEN']
print(f"HF Token retrieved: {HF_TOKEN[:8]}...")

In [ ]:
from huggingface_hub import snapshot_download

model_dir = "/tmp/medsiglip-448"
os.makedirs(model_dir, exist_ok=True)

snapshot_path = snapshot_download(
    repo_id="google/medsiglip-448",
    cache_dir=model_dir,
    token=HF_TOKEN
)
print(f"Model downloaded to: {snapshot_path}")
print(f"Contents: {os.listdir(snapshot_path)}")

In [ ]:
from snowflake.ml.model import custom_model
import pandas as pd
import numpy as np
import base64
from io import BytesIO


class MedSigLIPModel(custom_model.CustomModel):
    """Dual-encoder wrapping google/medsiglip-448 for medical image AND text embeddings."""

    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        import torch
        from transformers import AutoModel, AutoProcessor

        model_path = self.context.path("model_dir")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = AutoModel.from_pretrained(model_path).to(self.device).eval()
        self.processor = AutoProcessor.from_pretrained(model_path)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """Generate image embeddings from base64-encoded images.
        Input: DataFrame with 'IMAGE_BYTES' column (base64-encoded PNG/JPEG).
        Output: DataFrame with 'EMBEDDING' column (list of 1152 floats).
        """
        import torch
        from PIL import Image
        from io import BytesIO
        import base64

        embeddings_list = []
        for _, row in input_df.iterrows():
            img_b64 = row["IMAGE_BYTES"]
            img_data = base64.b64decode(img_b64)
            image = Image.open(BytesIO(img_data)).convert("RGB")

            inputs = self.processor(images=image, return_tensors="pt").to(self.device)
            with torch.no_grad():
                image_features = self.model.get_image_features(**inputs)

            embedding = image_features.cpu().numpy().flatten().tolist()
            embeddings_list.append(embedding)

        return pd.DataFrame({"EMBEDDING": embeddings_list})

    @custom_model.inference_api
    def embed_text(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """Generate text embeddings for natural language queries.
        Input: DataFrame with 'TEXT' column.
        Output: DataFrame with 'EMBEDDING' column (list of 1152 floats).
        """
        import torch

        embeddings_list = []
        for _, row in input_df.iterrows():
            text = row["TEXT"]
            inputs = self.processor(text=text, return_tensors="pt", padding="max_length").to(self.device)
            with torch.no_grad():
                text_features = self.model.get_text_features(**inputs)

            embedding = text_features.cpu().numpy().flatten().tolist()
            embeddings_list.append(embedding)

        return pd.DataFrame({"EMBEDDING": embeddings_list})

print("MedSigLIPModel class defined with predict() and embed_text() endpoints.")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

from PIL import Image
import base64
from io import BytesIO

test_img = Image.new("RGB", (448, 448), color=(128, 128, 128))
buf = BytesIO()
test_img.save(buf, format="PNG")
img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

mc = custom_model.ModelContext(model_dir=snapshot_path)
medsiglip = MedSigLIPModel(mc)
print(f"\nModel running on device: {medsiglip.device}")

# Test image embedding
test_input = pd.DataFrame({"IMAGE_BYTES": [img_b64]})
img_result = medsiglip.predict(test_input)
print(f"\nImage embedding: {len(img_result['EMBEDDING'][0])} dims")

# Test text embedding
test_text = pd.DataFrame({"TEXT": ["cardiac CT angiography with calcification"]})
txt_result = medsiglip.embed_text(test_text)
print(f"Text embedding: {len(txt_result['EMBEDDING'][0])} dims")

# Verify both in same vector space
img_emb = np.array(img_result['EMBEDDING'][0])
txt_emb = np.array(txt_result['EMBEDDING'][0])
cosine_sim = np.dot(img_emb, txt_emb) / (np.linalg.norm(img_emb) * np.linalg.norm(txt_emb))
print(f"\nCosine similarity (gray image vs 'cardiac CT angiography'): {cosine_sim:.4f}")
print("(Low similarity expected for a blank gray image — confirms dual encoder works)")

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import model_signature

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")

try:
    session.sql("DROP SERVICE IF EXISTS SF_CLINICAL_DB.UTILS.MEDSIGLIP_448_SVC").collect()
    session.sql("DROP MODEL IF EXISTS SF_CLINICAL_DB.UTILS.MEDSIGLIP_448").collect()
    print("Dropped existing service/model.")
except Exception as e:
    print(f"Cleanup: {e}")

test_text_input = pd.DataFrame({"TEXT": ["cardiac CT angiography"]})

signatures = {
    "predict": model_signature.infer_signature(
        input_data=test_input,
        output_data=img_result,
    ),
    "embed_text": model_signature.infer_signature(
        input_data=test_text_input,
        output_data=txt_result,
    ),
}

model_version = reg.log_model(
    model=medsiglip,
    model_name="MEDSIGLIP_448",
    version_name="V2",
    pip_requirements=[
        "transformers",
        "torch",
        "pillow",
        "huggingface_hub",
        "accelerate",
        "sentencepiece",
        "protobuf",
    ],
    signatures=signatures,
    options={
        "cuda_version": "12.3",
    },
    comment="MedSigLIP-448 dual encoder: predict(IMAGE_BYTES)->embeddings, embed_text(TEXT)->embeddings",
)

print(f"Model logged: {model_version.model_name} / {model_version.version_name}")
print("Endpoints: predict (images), embed_text (text queries)")

In [ ]:
model_version.create_service(
    service_name="MEDSIGLIP_448_SVC",
    service_compute_pool="EW_THV_COMPUTE_POOL",
    gpu_requests="1",
    max_instances=1,
)

print("Service deployment initiated.")
print("Check status: CALL SYSTEM$GET_SERVICE_STATUS('SF_CLINICAL_DB.UTILS.MEDSIGLIP_448_SVC');")

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")
mv = reg.get_model("MEDSIGLIP_448").version("V2")

inference_result = mv.run(
    test_input,
    function_name="predict",
    service_name="MEDSIGLIP_448_SVC",
)

print("Inference through SPCS service succeeded!")
print(f"Result embedding dimension: {len(inference_result['EMBEDDING'][0])}")
print(f"First 5 values: {inference_result['EMBEDDING'][0][:5]}")

## Phase 1: UDTF — Convert DICOM files to Base64 PNG (Warehouse-Parallel)
Creates a Python UDTF that reads DICOM files from the external stage via `SnowflakeFile`, applies windowing, normalizes, resizes to 448x448, and returns base64-encoded PNG. The warehouse parallelizes this across all files automatically.

In [ ]:
%%sql
CREATE OR REPLACE FUNCTION SF_CLINICAL_DB.UTILS.DICOM_TO_BASE64_PNG(file_path VARCHAR)
RETURNS TABLE (IMAGE_BYTES VARCHAR)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'pydicom', 'pillow', 'numpy')
HANDLER = 'DicomToBase64'
AS $$
import numpy as np
import base64
from io import BytesIO

class DicomToBase64:
    def process(self, file_path: str):
        import pydicom
        from PIL import Image
        from snowflake.snowpark.files import SnowflakeFile

        with SnowflakeFile.open(file_path, 'rb') as f:
            ds = pydicom.dcmread(f)

        pixel_array = ds.pixel_array.astype(np.float32)

        # Apply windowing if available
        if hasattr(ds, 'WindowCenter') and hasattr(ds, 'WindowWidth'):
            wc = ds.WindowCenter
            ww = ds.WindowWidth
            if hasattr(wc, '__iter__'):
                wc = wc[0]
            if hasattr(ww, '__iter__'):
                ww = ww[0]
            wc, ww = float(wc), float(ww)
            lower = wc - ww / 2
            upper = wc + ww / 2
            pixel_array = np.clip(pixel_array, lower, upper)

        # Normalize to 0-255
        pmin, pmax = pixel_array.min(), pixel_array.max()
        if pmax > pmin:
            pixel_array = ((pixel_array - pmin) / (pmax - pmin) * 255).astype(np.uint8)
        else:
            pixel_array = np.zeros_like(pixel_array, dtype=np.uint8)

        # Convert to RGB and resize to 448x448
        img = Image.fromarray(pixel_array).convert('RGB')
        img = img.resize((448, 448))

        buf = BytesIO()
        img.save(buf, format='PNG')
        img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')

        yield (img_b64,)
$$;

In [ ]:
%%sql
-- Test the UDTF on a single file
SELECT
    d.RELATIVE_PATH,
    LENGTH(t.IMAGE_BYTES) AS BASE64_LENGTH
FROM DIRECTORY(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG) d,
    TABLE(SF_CLINICAL_DB.UTILS.DICOM_TO_BASE64_PNG(
        BUILD_SCOPED_FILE_URL(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG, d.RELATIVE_PATH)
    )) t
WHERE d.RELATIVE_PATH LIKE '7e1bcefd-d097-47c0-9d54-65bd5d9674cf/%.dcm'
LIMIT 1;

In [ ]:
%%sql
-- Run UDTF on all DICOM files and persist base64 images to staging table
CREATE OR REPLACE TABLE SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES AS
WITH uuid_map AS (
    SELECT column1 AS UUID, column2 AS COLLECTION, column3 AS LABEL
    FROM VALUES
        ('7e1bcefd-d097-47c0-9d54-65bd5d9674cf', 'nsclc_radiogenomics', 'Cardiac CTA Diastolic 70% AMC-015'),
        ('3b5970f6-f130-4337-9666-a9741a267268', 'varepop_apollo', 'Cardiac CTA Systolic 31% AP-26JK'),
        ('4bed8f9a-bf84-44da-ac8d-d39e15d9c978', 'nsclc_radiogenomics', 'Cardiac CT Systolic 38% AMC-027'),
        ('f53b78ea-92ea-479e-a89f-967bec74e848', 'rider_lung_pet_ct', 'Gated Segment 0.625mm RIDER-2019259100'),
        ('1d92626a-e39d-4cf5-8ed9-864aae909d3d', 'rider_lung_pet_ct', 'Cine 30-65 BPM RIDER-2266952716'),
        ('7f4806f3-5946-41cb-8684-8fd5d51f8931', 'covid_19_ny_sbu', 'Cardiac 2.5 B41s A130302'),
        ('6dfa81e0-3520-4332-aecc-0767dd1cab74', 'covid_19_ny_sbu', 'CTA 3.0 CE A095019'),
        ('f6fc9ace-5516-418d-b979-7ffaf14de22b', 'nlst', 'Chest CT Philips 202207'),
        ('bb721c76-8056-4780-97be-1bc61b469454', 'nlst', 'Chest CT Siemens B30f 133417'),
        ('de9868a4-a9d9-4652-9f0f-04e06a95d09e', 'nlst', 'Chest CT Toshiba FC10 200119')
)
SELECT
    d.RELATIVE_PATH AS FILE_NAME,
    d.SIZE AS FILE_SIZE,
    SPLIT_PART(d.RELATIVE_PATH, '/', 1) AS SERIES_UUID,
    m.LABEL,
    m.COLLECTION,
    t.IMAGE_BYTES
FROM DIRECTORY(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG) d
JOIN uuid_map m ON SPLIT_PART(d.RELATIVE_PATH, '/', 1) = m.UUID,
    TABLE(SF_CLINICAL_DB.UTILS.DICOM_TO_BASE64_PNG(
        BUILD_SCOPED_FILE_URL(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG, d.RELATIVE_PATH)
    )) t
WHERE d.RELATIVE_PATH LIKE '%.dcm';

## Phase 2: Server-Side Inference via SPCS
Run `mv.run()` on the full Snowpark DataFrame — data stays server-side with no roundtrip to the notebook. Join with metadata and persist with `VECTOR(FLOAT, 1152)` type.

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")
mv = reg.get_model("MEDSIGLIP_448").version("V2")

total_rows = session.sql("SELECT COUNT(*) AS CNT FROM SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES").collect()[0]['CNT']
print(f"Total images to embed: {total_rows}")

input_sp_df = session.table("SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES").select("IMAGE_BYTES")

print("Running inference via SPCS service (server-side, no data roundtrip)...")
result_df = mv.run(
    input_sp_df,
    function_name="predict",
    service_name="MEDSIGLIP_448_SVC",
)

print(f"Inference complete! Result type: {type(result_df)}")
print(f"Result columns: {result_df.columns}")
result_df.show(5)

In [ ]:
from snowflake.snowpark import functions as F

source_df = session.table("SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES")

joined_df = source_df.join(
    result_df,
    source_df["IMAGE_BYTES"] == result_df["IMAGE_BYTES"],
    "inner"
).select(
    source_df["FILE_NAME"],
    source_df["FILE_SIZE"],
    source_df["SERIES_UUID"],
    source_df["LABEL"],
    source_df["COLLECTION"],
    result_df["EMBEDDING"]
)

print(f"Joined DataFrame schema:")
for field in joined_df.schema.fields:
    print(f"  {field.name}: {field.datatype}")
print(f"\nRow count: {joined_df.count()}")

In [ ]:
joined_df.create_or_replace_temp_view("_TMP_DICOM_EMBEDDINGS_VIEW")

session.sql("""
CREATE OR REPLACE TABLE SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS AS
SELECT
    FILE_NAME,
    FILE_SIZE,
    SERIES_UUID,
    LABEL,
    COLLECTION,
    EMBEDDING::VECTOR(FLOAT, 1152) AS EMBEDDING
FROM _TMP_DICOM_EMBEDDINGS_VIEW
""").collect()

print(f"Written {total_rows} rows to SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS with VECTOR(FLOAT, 1152).")

In [ ]:
verify = session.sql("""
    SELECT 
        COUNT(*) AS TOTAL_ROWS,
        COUNT(DISTINCT SERIES_UUID) AS UNIQUE_SERIES,
        COUNT(DISTINCT COLLECTION) AS UNIQUE_COLLECTIONS
    FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
""").to_pandas()
print("Table stats:")
print(verify.to_string(index=False))

print("\nSample rows with VECTOR verification:")
sample = session.sql("""
    SELECT LABEL, COLLECTION, TYPEOF(EMBEDDING) AS EMB_TYPE,
           VECTOR_L2_DISTANCE(EMBEDDING, EMBEDDING) AS SELF_DIST
    FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
    LIMIT 5
""").to_pandas()
print(sample.to_string(index=False))

## Semantic Search: Text-to-Image Retrieval
Embed a natural language query with `embed_text`, then find the most similar DICOM images using `VECTOR_COSINE_SIMILARITY`.

In [ ]:
from snowflake.ml.registry import Registry
import pandas as pd
import json

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")
mv = reg.get_model("MEDSIGLIP_448").version("V2")

query = "cardiac CT angiography with contrast"

query_input = pd.DataFrame({"TEXT": [query]})
query_result = mv.run(
    query_input,
    function_name="embed_text",
    service_name="MEDSIGLIP_448_SVC",
)

query_embedding = query_result['EMBEDDING'][0]
print(f"Query: '{query}'")
print(f"Embedded to {len(query_embedding)}-dim vector")
print(f"First 3 values: {query_embedding[:3]}")

In [ ]:
query_emb_str = json.dumps(query_embedding if isinstance(query_embedding, list) else query_embedding.tolist())

search_results = session.sql(f"""
    SELECT
        LABEL,
        COLLECTION,
        SERIES_UUID,
        FILE_NAME,
        VECTOR_COSINE_SIMILARITY(EMBEDDING, {query_emb_str}::VECTOR(FLOAT, 1152)) AS SIMILARITY
    FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
    ORDER BY SIMILARITY DESC
    LIMIT 10
""").to_pandas()

print(f"\nTop 10 results for: '{query}'")
print("=" * 80)
for i, row in search_results.iterrows():
    print(f"  {i+1}. [{row['SIMILARITY']:.4f}] {row['LABEL']} ({row['COLLECTION']})")
    print(f"     File: {row['FILE_NAME']}")
    print()

M Log google/medsiglip-448 to Snowflake Model Registry as a CustomModel for DICOM image embeddings
*Co-authored with CoCo*

In [ ]:
%%sql -r dataframe_1
USE SCHEMA SF_CLINICAL_DB.UTILS;

In [ ]:
%%sql -r dataframe_2
show secrets in database cortex_ai_db;

In [ ]:
# Install required packages
!pip install transformers torch pillow huggingface_hub --quiet

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Get Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Create a UDF to retrieve the HuggingFace token from the Snowflake secret
session.sql("""
CREATE OR REPLACE FUNCTION SF_CLINICAL_DB.UTILS.GET_HF_TOKEN()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'get_token'
EXTERNAL_ACCESS_INTEGRATIONS = (ALLOW_ALL_EAI)
SECRETS = ('hf_token' = CORTEX_AI_DB.UTILS.SECRET_NAME_HUGGING_FACE_TOKEN)
AS $$
import _snowflake
def get_token():
    return _snowflake.get_generic_secret_string('hf_token')
$$
""").collect()

# Retrieve the token
HF_TOKEN = session.sql("SELECT SF_CLINICAL_DB.UTILS.GET_HF_TOKEN() AS TOKEN").collect()[0]['TOKEN']
print(f"HF Token retrieved: {HF_TOKEN[:8]}...")

In [ ]:
from huggingface_hub import snapshot_download

# Download the MedSigLIP-448 model locally
model_dir = "/tmp/medsiglip-448"
os.makedirs(model_dir, exist_ok=True)

snapshot_path = snapshot_download(
    repo_id="google/medsiglip-448",
    cache_dir=model_dir,
    token=HF_TOKEN
)
print(f"Model downloaded to: {snapshot_path}")
print(f"Contents: {os.listdir(snapshot_path)}")

In [ ]:
from snowflake.ml.model import custom_model
import pandas as pd
import numpy as np
import base64
from io import BytesIO


class MedSigLIPModel(custom_model.CustomModel):
    """Dual-encoder wrapping google/medsiglip-448 for medical image AND text embeddings."""

    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        import torch
        from transformers import AutoModel, AutoProcessor

        model_path = self.context.path("model_dir")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = AutoModel.from_pretrained(model_path).to(self.device).eval()
        # Use full AutoProcessor (image processor + tokenizer) for dual-encoder
        self.processor = AutoProcessor.from_pretrained(model_path)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """Generate image embeddings from base64-encoded images.
        Input: DataFrame with 'IMAGE_BYTES' column (base64-encoded PNG/JPEG).
        Output: DataFrame with 'EMBEDDING' column (list of 1152 floats).
        """
        import torch
        from PIL import Image
        from io import BytesIO
        import base64

        embeddings_list = []
        for _, row in input_df.iterrows():
            img_b64 = row["IMAGE_BYTES"]
            img_data = base64.b64decode(img_b64)
            image = Image.open(BytesIO(img_data)).convert("RGB")

            inputs = self.processor(images=image, return_tensors="pt").to(self.device)
            with torch.no_grad():
                image_features = self.model.get_image_features(**inputs)

            embedding = image_features.cpu().numpy().flatten().tolist()
            embeddings_list.append(embedding)

        return pd.DataFrame({"EMBEDDING": embeddings_list})

    @custom_model.inference_api
    def embed_text(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """Generate text embeddings for search queries.
        Input: DataFrame with 'TEXT' column (natural language queries, max 64 tokens).
        Output: DataFrame with 'EMBEDDING' column (list of 1152 floats).
        """
        import torch

        embeddings_list = []
        for _, row in input_df.iterrows():
            text = row["TEXT"]
            inputs = self.processor(
                text=text, return_tensors="pt",
                padding=True, truncation=True, max_length=64
            ).to(self.device)

            with torch.no_grad():
                text_features = self.model.get_text_features(**inputs)

            embedding = text_features.cpu().numpy().flatten().tolist()
            embeddings_list.append(embedding)

        return pd.DataFrame({"EMBEDDING": embeddings_list})

print("MedSigLIPModel class defined with dual-encoder (predict + embed_text).")
print("  - predict(IMAGE_BYTES) -> image embeddings")
print("  - embed_text(TEXT) -> text embeddings (same vector space)")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# Create a test image
from PIL import Image
import base64
from io import BytesIO

test_img = Image.new("RGB", (448, 448), color=(128, 128, 128))
buf = BytesIO()
test_img.save(buf, format="PNG")
img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

# Create ModelContext and instantiate
mc = custom_model.ModelContext(model_dir=snapshot_path)
medsiglip = MedSigLIPModel(mc)
print(f"\nModel running on device: {medsiglip.device}")

# Test image embedding
test_input = pd.DataFrame({"IMAGE_BYTES": [img_b64]})
img_result = medsiglip.predict(test_input)
print(f"\nImage embedding: {len(img_result['EMBEDDING'][0])} dims")

# Test text embedding
test_text = pd.DataFrame({"TEXT": ["cardiac CT angiography with calcification"]})
txt_result = medsiglip.embed_text(test_text)
print(f"Text embedding: {len(txt_result['EMBEDDING'][0])} dims")

# Verify both are in the same vector space (compute cosine similarity)
import numpy as np
img_emb = np.array(img_result['EMBEDDING'][0])
txt_emb = np.array(txt_result['EMBEDDING'][0])
cosine_sim = np.dot(img_emb, txt_emb) / (np.linalg.norm(img_emb) * np.linalg.norm(txt_emb))
print(f"\nCosine similarity (gray image vs 'cardiac CT angiography'): {cosine_sim:.4f}")
print("(Low similarity expected for a blank gray image — confirms dual encoder works)")

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import model_signature

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")

# Drop existing model (already dropped service+model above, but just in case)
try:
    session.sql("DROP SERVICE IF EXISTS SF_CLINICAL_DB.UTILS.MEDSIGLIP_448_SVC").collect()
    session.sql("DROP MODEL IF EXISTS SF_CLINICAL_DB.UTILS.MEDSIGLIP_448").collect()
    print("Dropped existing service/model.")
except Exception as e:
    print(f"Cleanup: {e}")

# Define explicit signatures for both methods
test_text_input = pd.DataFrame({"TEXT": ["cardiac CT angiography"]})

signatures = {
    "predict": model_signature.infer_signature(
        input_data=test_input,
        output_data=img_result,
    ),
    "embed_text": model_signature.infer_signature(
        input_data=test_text_input,
        output_data=txt_result,
    ),
}

model_version = reg.log_model(
    model=medsiglip,
    model_name="MEDSIGLIP_448",
    version_name="V2",
    pip_requirements=[
        "transformers",
        "torch",
        "pillow",
        "huggingface_hub",
        "accelerate",
        "sentencepiece",
        "protobuf",
    ],
    signatures=signatures,
    options={
        "cuda_version": "12.3",
    },
    comment="MedSigLIP-448 dual encoder: predict(IMAGE_BYTES)->embeddings, embed_text(TEXT)->embeddings",
)

print(f"Model logged: {model_version.model_name} / {model_version.version_name}")
print("Endpoints: predict (images), embed_text (text queries)")

In [ ]:
# Deploy as inference service on GPU compute pool
model_version.create_service(
    service_name="MEDSIGLIP_448_SVC",
    service_compute_pool="EW_THV_COMPUTE_POOL",
    gpu_requests="1",
    max_instances=1,
)

print("Service deployment initiated.")
print("Check status: CALL SYSTEM$GET_SERVICE_STATUS('SF_CLINICAL_DB.UTILS.MEDSIGLIP_448_SVC');")

In [ ]:
# Test inference through the deployed SPCS service
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")
mv = reg.get_model("MEDSIGLIP_448").version("V1")

inference_result = mv.run(
    test_input,
    function_name="predict",
    service_name="MEDSIGLIP_448_SVC",
)

print("Inference through SPCS service succeeded!")
print(f"Result embedding dimension: {len(inference_result['EMBEDDING'][0])}")
print(f"First 5 values: {inference_result['EMBEDDING'][0][:5]}")

In [ ]:
test_input

## Inference on 10 Cardiac DICOM Samples
Query DICOM files from stage, convert pixel data to base64 PNG, and generate embeddings via the SPCS service.

In [ ]:
# Query 10 cardiac DICOM records - one file per series
dicom_query = """
WITH uuid_map AS (
    SELECT column1 AS UUID, column2 AS COLLECTION, column3 AS LABEL
    FROM VALUES
        ('7e1bcefd-d097-47c0-9d54-65bd5d9674cf', 'nsclc_radiogenomics', 'Cardiac CTA Diastolic 70% AMC-015'),
        ('3b5970f6-f130-4337-9666-a9741a267268', 'varepop_apollo', 'Cardiac CTA Systolic 31% AP-26JK'),
        ('4bed8f9a-bf84-44da-ac8d-d39e15d9c978', 'nsclc_radiogenomics', 'Cardiac CT Systolic 38% AMC-027'),
        ('f53b78ea-92ea-479e-a89f-967bec74e848', 'rider_lung_pet_ct', 'Gated Segment 0.625mm RIDER-2019259100'),
        ('1d92626a-e39d-4cf5-8ed9-864aae909d3d', 'rider_lung_pet_ct', 'Cine 30-65 BPM RIDER-2266952716'),
        ('7f4806f3-5946-41cb-8684-8fd5d51f8931', 'covid_19_ny_sbu', 'Cardiac 2.5 B41s A130302'),
        ('6dfa81e0-3520-4332-aecc-0767dd1cab74', 'covid_19_ny_sbu', 'CTA 3.0 CE A095019'),
        ('f6fc9ace-5516-418d-b979-7ffaf14de22b', 'nlst', 'Chest CT Philips 202207'),
        ('bb721c76-8056-4780-97be-1bc61b469454', 'nlst', 'Chest CT Siemens B30f 133417'),
        ('de9868a4-a9d9-4652-9f0f-04e06a95d09e', 'nlst', 'Chest CT Toshiba FC10 200119')
),
raw_files AS (
    SELECT
        RELATIVE_PATH AS FILE_NAME,
        SIZE AS FILE_SIZE,
        SPLIT_PART(RELATIVE_PATH, '/', 1) AS SERIES_UUID
    FROM DIRECTORY(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG)
    WHERE RELATIVE_PATH LIKE '%.dcm'
),
matched AS (
    SELECT f.FILE_NAME, f.FILE_SIZE, f.SERIES_UUID, m.LABEL, m.COLLECTION
    FROM raw_files f
    JOIN uuid_map m ON f.SERIES_UUID = m.UUID
)
SELECT *
FROM matched
QUALIFY ROW_NUMBER() OVER (PARTITION BY SERIES_UUID ORDER BY FILE_NAME) = 1
ORDER BY COLLECTION, LABEL
"""

dicom_df = session.sql(dicom_query).to_pandas()
print(f"Found {len(dicom_df)} DICOM files (1 per series):")
print(dicom_df[['SERIES_UUID', 'LABEL', 'COLLECTION']].to_string(index=False))

In [ ]:
!pip install pydicom --quiet

In [ ]:
import pydicom
import base64
import numpy as np
from PIL import Image
from io import BytesIO
import tempfile

def dicom_to_base64_png(file_path: str) -> str:
    """Read a DICOM file and convert its pixel data to a base64-encoded PNG."""
    ds = pydicom.dcmread(file_path)
    pixel_array = ds.pixel_array.astype(np.float32)

    # Apply windowing if available
    if hasattr(ds, 'WindowCenter') and hasattr(ds, 'WindowWidth'):
        wc = float(ds.WindowCenter if not isinstance(ds.WindowCenter, pydicom.multival.MultiValue) else ds.WindowCenter[0])
        ww = float(ds.WindowWidth if not isinstance(ds.WindowWidth, pydicom.multival.MultiValue) else ds.WindowWidth[0])
        lower = wc - ww / 2
        upper = wc + ww / 2
        pixel_array = np.clip(pixel_array, lower, upper)

    # Normalize to 0-255
    pmin, pmax = pixel_array.min(), pixel_array.max()
    if pmax > pmin:
        pixel_array = ((pixel_array - pmin) / (pmax - pmin) * 255).astype(np.uint8)
    else:
        pixel_array = np.zeros_like(pixel_array, dtype=np.uint8)

    # Convert to RGB PIL image
    img = Image.fromarray(pixel_array).convert("RGB")
    img = img.resize((448, 448))  # MedSigLIP expects 448x448

    buf = BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

print("DICOM-to-PNG conversion function ready.")

In [ ]:
# Download DICOM files directly from public S3 bucket (idc-open-data is public)
import os
import requests

images_b64 = []
labels = []
failed = []

S3_BASE = "https://idc-open-data.s3.amazonaws.com"

for idx, row in dicom_df.iterrows():
    file_name = row['FILE_NAME']
    label = row['LABEL']

    # IDC public bucket - direct HTTP access
    url = f"{S3_BASE}/{file_name}"
    resp = requests.get(url)

    if resp.status_code != 200:
        print(f"  [x] FAILED to download {label}: HTTP {resp.status_code}")
        failed.append((label, f"HTTP {resp.status_code}"))
        continue

    # Save to temp file for pydicom
    local_path = f"/tmp/dicom_{idx}.dcm"
    with open(local_path, 'wb') as f:
        f.write(resp.content)

    try:
        img_b64 = dicom_to_base64_png(local_path)
        images_b64.append(img_b64)
        labels.append(label)
        print(f"  [{len(images_b64)}/10] Converted: {label}")
    except Exception as e:
        failed.append((label, str(e)))
        print(f"  [x] FAILED {label}: {e}")
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)

print(f"\nSuccessfully converted {len(images_b64)} DICOM images to base64 PNG.")
if failed:
    print(f"Failed: {len(failed)} images")

In [ ]:
# Run inference on all converted images through the SPCS service
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")
mv = reg.get_model("MEDSIGLIP_448").version("V1")

inference_input = pd.DataFrame({"IMAGE_BYTES": images_b64})

print(f"Running inference on {len(inference_input)} images via SPCS service...")
inference_result = mv.run(
    inference_input,
    function_name="predict",
    service_name="MEDSIGLIP_448_SVC",
)

print(f"\nInference complete! Results:")
for i, label in enumerate(labels):
    emb = inference_result['EMBEDDING'][i]
    print(f"  {label}: {len(emb)}-dim embedding, first 3 values = {emb[:3]}")

## Phase 1: UDTF — Convert 23k DICOM files to Base64 PNG (Warehouse-Parallel)
Creates a Python UDTF that reads DICOM files from the external stage via `SnowflakeFile`, applies windowing, normalizes, resizes to 448x448, and returns base64-encoded PNG. The warehouse parallelizes this across all files automatically.

In [ ]:
%%sql -r create_udtf_result
-- Create the DICOM-to-base64 conversion UDTF
CREATE OR REPLACE FUNCTION SF_CLINICAL_DB.UTILS.DICOM_TO_BASE64_PNG(file_path VARCHAR)
RETURNS TABLE (IMAGE_BYTES VARCHAR)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'pydicom', 'pillow', 'numpy')
HANDLER = 'DicomToBase64'
AS $$
import numpy as np
import base64
from io import BytesIO

class DicomToBase64:
    def process(self, file_path: str):
        import pydicom
        from PIL import Image
        from snowflake.snowpark.files import SnowflakeFile

        with SnowflakeFile.open(file_path, 'rb') as f:
            ds = pydicom.dcmread(f)

        pixel_array = ds.pixel_array.astype(np.float32)

        # Apply windowing if available
        if hasattr(ds, 'WindowCenter') and hasattr(ds, 'WindowWidth'):
            wc = ds.WindowCenter
            ww = ds.WindowWidth
            if hasattr(wc, '__iter__'):
                wc = wc[0]
            if hasattr(ww, '__iter__'):
                ww = ww[0]
            wc, ww = float(wc), float(ww)
            lower = wc - ww / 2
            upper = wc + ww / 2
            pixel_array = np.clip(pixel_array, lower, upper)

        # Normalize to 0-255
        pmin, pmax = pixel_array.min(), pixel_array.max()
        if pmax > pmin:
            pixel_array = ((pixel_array - pmin) / (pmax - pmin) * 255).astype(np.uint8)
        else:
            pixel_array = np.zeros_like(pixel_array, dtype=np.uint8)

        # Convert to RGB and resize to 448x448
        img = Image.fromarray(pixel_array).convert('RGB')
        img = img.resize((448, 448))

        buf = BytesIO()
        img.save(buf, format='PNG')
        img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')

        yield (img_b64,)
$$;

In [ ]:
%%sql -r udtf_test
-- Test the UDTF on a single file to verify it works
SELECT
    d.RELATIVE_PATH,
    LENGTH(t.IMAGE_BYTES) AS BASE64_LENGTH
FROM DIRECTORY(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG) d,
    TABLE(SF_CLINICAL_DB.UTILS.DICOM_TO_BASE64_PNG(
        BUILD_SCOPED_FILE_URL(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG, d.RELATIVE_PATH)
    )) t
WHERE d.RELATIVE_PATH LIKE '7e1bcefd-d097-47c0-9d54-65bd5d9674cf/%.dcm'
LIMIT 1;

In [ ]:
%%sql -r ctas_result
-- Phase 1: Run UDTF on all 23k DICOM files and persist to table
CREATE OR REPLACE TABLE SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES AS
WITH uuid_map AS (
    SELECT column1 AS UUID, column2 AS COLLECTION, column3 AS LABEL
    FROM VALUES
        ('7e1bcefd-d097-47c0-9d54-65bd5d9674cf', 'nsclc_radiogenomics', 'Cardiac CTA Diastolic 70% AMC-015'),
        ('3b5970f6-f130-4337-9666-a9741a267268', 'varepop_apollo', 'Cardiac CTA Systolic 31% AP-26JK'),
        ('4bed8f9a-bf84-44da-ac8d-d39e15d9c978', 'nsclc_radiogenomics', 'Cardiac CT Systolic 38% AMC-027'),
        ('f53b78ea-92ea-479e-a89f-967bec74e848', 'rider_lung_pet_ct', 'Gated Segment 0.625mm RIDER-2019259100'),
        ('1d92626a-e39d-4cf5-8ed9-864aae909d3d', 'rider_lung_pet_ct', 'Cine 30-65 BPM RIDER-2266952716'),
        ('7f4806f3-5946-41cb-8684-8fd5d51f8931', 'covid_19_ny_sbu', 'Cardiac 2.5 B41s A130302'),
        ('6dfa81e0-3520-4332-aecc-0767dd1cab74', 'covid_19_ny_sbu', 'CTA 3.0 CE A095019'),
        ('f6fc9ace-5516-418d-b979-7ffaf14de22b', 'nlst', 'Chest CT Philips 202207'),
        ('bb721c76-8056-4780-97be-1bc61b469454', 'nlst', 'Chest CT Siemens B30f 133417'),
        ('de9868a4-a9d9-4652-9f0f-04e06a95d09e', 'nlst', 'Chest CT Toshiba FC10 200119')
)
SELECT
    d.RELATIVE_PATH AS FILE_NAME,
    d.SIZE AS FILE_SIZE,
    SPLIT_PART(d.RELATIVE_PATH, '/', 1) AS SERIES_UUID,
    m.LABEL,
    m.COLLECTION,
    t.IMAGE_BYTES
FROM DIRECTORY(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG) d
JOIN uuid_map m ON SPLIT_PART(d.RELATIVE_PATH, '/', 1) = m.UUID,
    TABLE(SF_CLINICAL_DB.UTILS.DICOM_TO_BASE64_PNG(
        BUILD_SCOPED_FILE_URL(@EW_IMAGING_DB.EXPLORER.IDC_OPEN_DATA_ALL_STG, d.RELATIVE_PATH)
    )) t
WHERE d.RELATIVE_PATH LIKE '%.dcm';

## Phase 2: Batched Inference via SPCS Service
Reads from `DICOM_BASE64_IMAGES` in chunks of 50, calls `mv.run()` on each batch, and writes embeddings to the final table.

In [ ]:
# Phase 2: Run inference server-side using mv.run() on Snowpark DataFrame
# This keeps all data on the Snowflake side - no roundtrip to the notebook
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")
mv = reg.get_model("MEDSIGLIP_448").version("V1")

# Get total count
total_rows = session.sql("SELECT COUNT(*) AS CNT FROM SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES").collect()[0]['CNT']
print(f"Total images to embed: {total_rows}")

# Run inference directly on a Snowpark DataFrame (stays server-side on SPCS)
# Pass only the IMAGE_BYTES column as the model expects
input_sp_df = session.table("SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES").select("IMAGE_BYTES")

print("Running inference via SPCS service (server-side, no data roundtrip)...")
result_df = mv.run(
    input_sp_df,
    function_name="predict",
    service_name="MEDSIGLIP_448_SVC",
)

print(f"Inference complete! Result type: {type(result_df)}")
print(f"Result columns: {result_df.columns}")
result_df.show(5)

In [ ]:
# Join embeddings with source metadata and persist to final table
from snowflake.snowpark import functions as F, Window

# Add row number to both source and result for a positional join
source_df = session.table("SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES").select(
    "FILE_NAME", "SERIES_UUID", "LABEL", "COLLECTION", "IMAGE_BYTES"
).with_column("ROW_ID", F.row_number().over(Window.order_by("FILE_NAME")))

result_with_id = result_df.with_column(
    "ROW_ID", F.row_number().over(Window.order_by("IMAGE_BYTES"))
)

# Join on IMAGE_BYTES (deterministic since they're unique per file)
final_df = source_df.join(
    result_with_id.select("IMAGE_BYTES", "EMBEDDING"),
    on="IMAGE_BYTES"
).select("FILE_NAME", "SERIES_UUID", "LABEL", "COLLECTION", "EMBEDDING")

# Write to table
final_df.write.mode("overwrite").save_as_table("SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS")

count = session.sql("SELECT COUNT(*) AS CNT FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS").collect()[0]['CNT']
print(f"Done! {count} embeddings persisted to SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS")

In [ ]:
%%sql -r verify_embeddings
-- Verify the embeddings table
SELECT
    COLLECTION,
    LABEL,
    COUNT(*) AS NUM_EMBEDDINGS,
    ARRAY_SIZE(EMBEDDING) AS EMBEDDING_DIM
FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
GROUP BY COLLECTION, LABEL, ARRAY_SIZE(EMBEDDING)
ORDER BY COLLECTION, LABEL;

In [ ]:
# Write embeddings to Snowflake table with VECTOR(FLOAT, 1152) data type
import json

# Create the target table with proper VECTOR column
session.sql("""
CREATE OR REPLACE TABLE SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS (
    SERIES_UUID VARCHAR,
    COLLECTION VARCHAR,
    LABEL VARCHAR,
    FILE_NAME VARCHAR,
    EMBEDDING VECTOR(FLOAT, 1152)
)
""").collect()
print("Created table SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS with VECTOR(FLOAT, 1152) column")

# Insert each row using SELECT ... cast to VECTOR
for i, label in enumerate(labels):
    row = dicom_df[dicom_df['LABEL'] == label].iloc[0]
    emb = inference_result['EMBEDDING'][i]
    emb_str = json.dumps(emb)

    session.sql(f"""
        INSERT INTO SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
        (SERIES_UUID, COLLECTION, LABEL, FILE_NAME, EMBEDDING)
        SELECT
            '{row['SERIES_UUID']}',
            '{row['COLLECTION']}',
            '{row['LABEL'].replace("'", "''")}',
            '{row['FILE_NAME'].replace("'", "''")}',
            {emb_str}::VECTOR(FLOAT, 1152)
    """).collect()

print(f"Inserted {len(labels)} rows with VECTOR embeddings.")

# Verify
verify = session.sql("""
    SELECT LABEL, TYPEOF(EMBEDDING) AS EMB_TYPE,
           VECTOR_L2_DISTANCE(EMBEDDING, EMBEDDING) AS SELF_DIST
    FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
    LIMIT 3
""").to_pandas()
print("\nVerification (TYPEOF + self-distance should be 0):")
print(verify.to_string(index=False))

## Phase 2: Full Inference on DICOM_BASE64_IMAGES
Run inference server-side using `mv.run()` on Snowpark DataFrame — keeps all data on the Snowflake side with no roundtrip to the notebook.

In [ ]:
# Phase 2: Run inference server-side using mv.run() on Snowpark DataFrame
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")
mv = reg.get_model("MEDSIGLIP_448").version("V1")

# Get total count
total_rows = session.sql("SELECT COUNT(*) AS CNT FROM SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES").collect()[0]['CNT']
print(f"Total images to embed: {total_rows}")

# Run inference directly on a Snowpark DataFrame (stays server-side on SPCS)
input_sp_df = session.table("SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES").select("IMAGE_BYTES")

print("Running inference via SPCS service (server-side, no data roundtrip)...")
result_df = mv.run(
    input_sp_df,
    function_name="predict",
    service_name="MEDSIGLIP_448_SVC",
)

print(f"Inference complete! Result type: {type(result_df)}")
print(f"Result columns: {result_df.columns}")
result_df.show(5)

In [ ]:
# Join result embeddings with source metadata using IMAGE_BYTES as join key
# Then write to table with VECTOR data type using CTAS
from snowflake.snowpark import functions as F

source_df = session.table("SF_CLINICAL_DB.UTILS.DICOM_BASE64_IMAGES")

# Join on IMAGE_BYTES to associate embeddings with metadata
joined_df = source_df.join(
    result_df,
    source_df["IMAGE_BYTES"] == result_df["IMAGE_BYTES"],
    "inner"
).select(
    source_df["FILE_NAME"],
    source_df["FILE_SIZE"],
    source_df["SERIES_UUID"],
    source_df["LABEL"],
    source_df["COLLECTION"],
    result_df["EMBEDDING"]
)

print(f"Joined DataFrame schema:")
for field in joined_df.schema.fields:
    print(f"  {field.name}: {field.datatype}")
print(f"\nRow count: {joined_df.count()}")

In [ ]:
# Write directly to final table with VECTOR cast — no staging table needed
# Use a temp view from the Snowpark DataFrame, then CTAS with inline cast
joined_df.create_or_replace_temp_view("_TMP_DICOM_EMBEDDINGS_VIEW")

session.sql("""
CREATE OR REPLACE TABLE SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS AS
SELECT
    FILE_NAME,
    FILE_SIZE,
    SERIES_UUID,
    LABEL,
    COLLECTION,
    EMBEDDING::VECTOR(FLOAT, 1152) AS EMBEDDING
FROM _TMP_DICOM_EMBEDDINGS_VIEW
""").collect()

print(f"Written {total_rows} rows to SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS with VECTOR(FLOAT, 1152) — no staging table.")

In [ ]:
# Verify the results
verify = session.sql("""
    SELECT 
        COUNT(*) AS TOTAL_ROWS,
        COUNT(DISTINCT SERIES_UUID) AS UNIQUE_SERIES,
        COUNT(DISTINCT COLLECTION) AS UNIQUE_COLLECTIONS
    FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
""").to_pandas()
print("Table stats:")
print(verify.to_string(index=False))

# Verify VECTOR type and self-distance
print("\nSample rows with VECTOR verification:")
sample = session.sql("""
    SELECT LABEL, COLLECTION, TYPEOF(EMBEDDING) AS EMB_TYPE,
           VECTOR_L2_DISTANCE(EMBEDDING, EMBEDDING) AS SELF_DIST
    FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
    LIMIT 5
""").to_pandas()
print(sample.to_string(index=False))

## Semantic Search: Text-to-Image Retrieval
Embed a natural language query with `embed_text`, then find the most similar DICOM images using `VECTOR_COSINE_SIMILARITY`.

In [ ]:
# Semantic search: embed a text query and find the most similar DICOM images
from snowflake.ml.registry import Registry
import pandas as pd
import json

reg = Registry(session=session, database_name="SF_CLINICAL_DB", schema_name="UTILS")
mv = reg.get_model("MEDSIGLIP_448").version("V2")

# --- Your search query ---
query = "cardiac CT angiography with contrast"

# Embed the text query via SPCS service
query_input = pd.DataFrame({"TEXT": [query]})
query_result = mv.run(
    query_input,
    function_name="embed_text",
    service_name="MEDSIGLIP_448_SVC",
)

query_embedding = query_result['EMBEDDING'][0]
print(f"Query: '{query}'")
print(f"Embedded to {len(query_embedding)}-dim vector")
print(f"First 3 values: {query_embedding[:3]}")

In [ ]:
# Search: find top-K most similar images using VECTOR_COSINE_SIMILARITY
query_emb_str = json.dumps(query_embedding if isinstance(query_embedding, list) else query_embedding.tolist())

search_results = session.sql(f"""
    SELECT
        LABEL,
        COLLECTION,
        SERIES_UUID,
        FILE_NAME,
        VECTOR_COSINE_SIMILARITY(EMBEDDING, {query_emb_str}::VECTOR(FLOAT, 1152)) AS SIMILARITY
    FROM SF_CLINICAL_DB.UTILS.DICOM_EMBEDDINGS
    ORDER BY SIMILARITY DESC
    LIMIT 10
""").to_pandas()

print(f"\nTop 10 results for: '{query}'")
print("=" * 80)
for i, row in search_results.iterrows():
    print(f"  {i+1}. [{row['SIMILARITY']:.4f}] {row['LABEL']} ({row['COLLECTION']})")
    print(f"     File: {row['FILE_NAME']}")
    print()